In [ ]:
#plink и bcftools
#в wsl
conda create -n plink
conda install -c bioconda plink

#конвертация в .vcf. Аутпут - snps_clean.vcf
plink --23file SNP_raw_v4_Full_20170514175358.txt --recode vcf --out snps_clean \
      --output-chr MT --snps-only just-acgt

#убрать позиции, идентичные референсу - оставить только snp. Аутпут - snps_filtered.vcf, 164638 SNP
conda install -c bioconda bcftools

bcftools view -i 'GT!="0/0"' snps_clean.vcf > snps_filtered.vcf

"Where do we come from?"
поиск митохондриальных SNP и SNP Y-хромосомы  
Y  
https://ytree.morleydna.com/processAutosomalExtract  
MT  
https://dna.jameslick.com/mthap/mthap.cgi  

In [ ]:
#"Who are we?"
#определение цвета глаз и пола
#есть SNP в Y - мужской пол
grep "^chrY\|^Y" snps_filtered.vcf | wc -l
#1032

#цвет глаз - гетерозигота A/G в гене OCA2/HERC2 - не голубые
grep "rs12913832" snps_filtered.vcf
#гетерозигота по rs16891982, rs12203592

In [ ]:
#поиск клинически значимых SNP
mkdir -p /root/.vep
cd /root/.vep
wget https://ftp.ensembl.org/pub/release-105/variation/indexed_vep_cache/homo_sapiens_vep_105_GRCh37.tar.gz
tar -xvzf homo_sapiens_vep_105_GRCh37.tar.gz

vep -i /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab5/snps_clean.vcf \
  --cache \
  --fork 8 \
  --assembly GRCh37 \
  --vcf \
  --symbol \
  --canonical \
  --protein \
  --custom clinvar.vcf.gz,ClinVar,vcf,exact,0,CLNSIG \
  --output_file /mnt/d/data/analysis_of_genomic_and_transcriptomic_data/lab5/vep_output_new1.vcf

#поиск SNP 
grep "risk_factor" vep_output_new1.vcf | cut -f1,3